# History of tesselations |  Manim Animation

This notebook builds the videos to create an animation-explainer of the history of tesselations.
It animates tiles or polygons to show how to build different tesselations.

In [1]:
import manim as mn
from manim import *

config.media_width = "75%"
config.verbosity = "WARNING"

print(mn.__version__)

0.21.0


## Intro to regular polygons

In [2]:
%%manim -qm PrimerosPoligonos

class PrimerosPoligonos(Scene):
    def construct(self):
        title = Text(
            "Estos son los 12 primeros polígonos regulares",
            font_size=30,
            color=WHITE,
        )
        if title.width > config.frame_width - 1:
            title.scale_to_fit_width(config.frame_width - 1)
        title.to_edge(UP, buff=0.45)

        colors = [
            "#FF8A80",  # triangle
            "#FFD180",  # square
            "#FFFF8D",  # pentagon
            "#B9F6CA",  # hexagon
            "#80D8FF",  # heptagon
            "#8C9EFF",  # octagon
            "#EA80FC",  # nonagon
            "#FF80AB",  # decagon
            "#CCFF90",  # hendecagon
            "#A7FFEB",  # dodecagon
        ]

        polys = VGroup()
        for n, color in zip(range(3, 13), colors):
            poly = RegularPolygon(
                n=n,
                radius=0.7,
                color=color,
                stroke_width=4,
                fill_color=color,
                fill_opacity=0.45,
            )
            if n == 4:
                poly.rotate(45 * DEGREES)
            polys.add(poly)

        polys.arrange_in_grid(rows=2, cols=5, buff=0.55)
        polys.next_to(title, DOWN, buff=0.7)
        polys.shift(DOWN * 0.15 * config.frame_height)

        self.play(FadeIn(title), run_time=0.8)
        self.wait(0.3)

        for poly in polys:
            self.play(Create(poly), run_time=0.45)
        self.wait(0.8)

        question = VGroup(
            Text(
                "¿Qué polígonos regulares permiten cubrir",
                font_size=32,
                color=WHITE,
            ),
            Text(
                "el plano sin dejar huecos?",
                font_size=32,
                color=WHITE,
            ),
        ).arrange(DOWN, buff=0.2)
        if question.width > config.frame_width - 1:
            question.scale_to_fit_width(config.frame_width - 1)
        question.to_edge(UP, buff=0.45)

        self.play(FadeOut(title), FadeIn(question), run_time=1)
        self.wait(2)

        answer = Text("Solo hay 3", font_size=40, color=WHITE)
        answer.to_edge(UP, buff=0.45)
        self.play(FadeOut(question), FadeIn(answer), run_time=1)
        self.wait(0.5)

        to_remove = VGroup(
            *[poly for n, poly in zip(range(3, 13), polys) if n not in (3, 4, 6)]
        )
        self.play(FadeOut(to_remove), run_time=1.2)
        self.wait(0.4)

        prefix = Text("Veamos como cubren el plano", font_size=32, color=WHITE)
        prefix.next_to(answer, DOWN, buff=0.45)
        prefix.shift(DOWN * 0.6 * config.frame_height)

        self.play(FadeIn(prefix), run_time=0.8)
        self.wait(2)
   

Manim Community v0.21.0

## Triangles

In [3]:
%%manim -qm TriangleTessellationFlyIn

class TriangleTessellationFlyIn(Scene):
    def construct(self):
        rows, cols = 3, 3  # 3 up+down pairs = 6 triangles per row
        radius = 1
        side = radius * (3**0.5)
        height = 1.5 * radius
        start_x = config.frame_width / 2 + side
        stroke_width = 4

        def centroid(mob):
            return mob.get_vertices().mean(axis=0)

        def move_centroid_to(mob, point):
            mob.shift(point - centroid(mob))

        def pair_centroids(r, c):
            x_up = c * side + r * (side / 2)
            y_up = -r * height
            up = RIGHT * x_up + UP * y_up
            down = RIGHT * (x_up + side / 2) + UP * (y_up + radius / 2)
            return [(up, True), (down, False)]

        def make_triangle(pointing_up):
            tri = RegularPolygon(
                n=3,
                radius=radius,
                color=WHITE,
                stroke_width=stroke_width,
            )
            if not pointing_up:
                tri.rotate(PI, about_point=centroid(tri))
            return tri

        initial = []
        for r in range(rows):
            for c in range(cols):
                initial.extend(pair_centroids(r, c))

        shift = -sum((t[0] for t in initial), ORIGIN) / len(initial)
        initial = [(pos + shift, up) for pos, up in initial]
        initial_keys = {(r, c) for r in range(rows) for c in range(cols)}

        label = Text("Triángulos", font_size=40, color="#E8D4A8")
        label.to_edge(UP, buff=0.4)
        self.play(FadeIn(label), run_time=0.6)

        for target, pointing_up in initial:
            tri = make_triangle(pointing_up)
            tri.rotate(90 * DEGREES, about_point=centroid(tri))

            start = RIGHT * start_x + UP * target[1]
            near = target + RIGHT * (1.2 * radius)
            move_centroid_to(tri, start)
            self.add(tri)

            self.play(
                tri.animate.shift(near - start),
                run_time=0.06,
                rate_func=rate_functions.ease_in_cubic,
            )

            tri.save_state()
            c0 = centroid(tri).copy()

            def land(mob, alpha, _c0=c0, _target=target):
                mob.restore()
                mob.shift(alpha * (_target - _c0))
                mob.rotate(
                    -90 * DEGREES * alpha,
                    about_point=centroid(mob),
                )

            self.play(
                UpdateFromAlphaFunc(tri, land),
                run_time=0.16,
                rate_func=rate_functions.ease_out_cubic,
            )

        margin_x = config.frame_width / 2 + 2 * side
        margin_y = config.frame_height / 2 + 2 * height
        rest = []
        for r in range(-8, 12):
            for c in range(-12, 16):
                if (r, c) in initial_keys:
                    continue
                for pos, pointing_up in pair_centroids(r, c):
                    pos = pos + shift
                    if abs(pos[0]) > margin_x or abs(pos[1]) > margin_y:
                        continue
                    tri = make_triangle(pointing_up)
                    move_centroid_to(tri, pos)
                    rest.append(tri)

        rest.sort(key=lambda mob: np.linalg.norm(centroid(mob)))
        self.play(FadeOut(label), run_time=0.4)
        self.play(
            LaggedStart(*[Create(tri) for tri in rest], lag_ratio=0.02),
            run_time=2,
        )
        self.wait()


Manim Community v0.21.0

## Squares

In [4]:
%%manim -qm SquareTessellationFlyIn

class SquareTessellationFlyIn(Scene):
    def construct(self):
        rows, cols = 3, 4
        radius = 1
        dx = 2 * radius
        dy = radius
        start_x = config.frame_width / 2 + dx
        stroke_width = 4
        tilt = 45 * DEGREES

        def centroid(mob):
            return mob.get_vertices().mean(axis=0)

        def move_centroid_to(mob, point):
            mob.shift(point - centroid(mob))

        def diamond_center(r, c):
            x = c * dx + (dx / 2 if r % 2 else 0)
            y = -r * dy
            return RIGHT * x + UP * y

        def make_diamond():
            return RegularPolygon(
                n=4,
                radius=radius,
                color=WHITE,
                stroke_width=stroke_width,
            )

        initial = [diamond_center(r, c) for r in range(rows) for c in range(cols)]
        shift = -sum(initial, ORIGIN) / len(initial)
        initial = [pos + shift for pos in initial]
        initial_keys = {(r, c) for r in range(rows) for c in range(cols)}

        label = Text("Cuadrados", font_size=40, color="#E8D4A8")
        label.to_edge(UP, buff=0.4)
        self.play(FadeIn(label), run_time=0.6)

        for target in initial:
            sq = make_diamond()
            sq.rotate(tilt, about_point=centroid(sq))

            start = RIGHT * start_x + UP * target[1]
            near = target + RIGHT * (1.2 * radius)
            move_centroid_to(sq, start)
            self.add(sq)

            self.play(
                sq.animate.shift(near - start),
                run_time=0.06,
                rate_func=rate_functions.ease_in_cubic,
            )

            sq.save_state()
            c0 = centroid(sq).copy()

            def land(mob, alpha, _c0=c0, _target=target):
                mob.restore()
                mob.shift(alpha * (_target - _c0))
                mob.rotate(
                    -tilt * alpha,
                    about_point=centroid(mob),
                )

            self.play(
                UpdateFromAlphaFunc(sq, land),
                run_time=0.16,
                rate_func=rate_functions.ease_out_cubic,
            )

        margin_x = config.frame_width / 2 + dx
        margin_y = config.frame_height / 2 + dx
        rest = []
        for r in range(-12, 16):
            for c in range(-12, 16):
                if (r, c) in initial_keys:
                    continue
                pos = diamond_center(r, c) + shift
                if abs(pos[0]) > margin_x or abs(pos[1]) > margin_y:
                    continue
                sq = make_diamond()
                move_centroid_to(sq, pos)
                rest.append(sq)

        rest.sort(key=lambda mob: np.linalg.norm(centroid(mob)))
        self.play(FadeOut(label), run_time=0.4)
        self.play(
            LaggedStart(*[Create(sq) for sq in rest], lag_ratio=0.02),
            run_time=2,
        )
        self.wait()


Manim Community v0.21.0

## Hexagons

In [5]:
%%manim -qm HexagonTessellationFlyIn

class HexagonTessellationFlyIn(Scene):
    def construct(self):
        rows, cols = 3, 3
        radius = 1
        dx = radius * (3**0.5)
        dy = 1.5 * radius
        start_x = config.frame_width / 2 + dx
        stroke_width = 4
        tilt = 30 * DEGREES

        def centroid(mob):
            return mob.get_vertices().mean(axis=0)

        def move_centroid_to(mob, point):
            mob.shift(point - centroid(mob))

        def hex_center(r, c):
            x = c * dx + (dx / 2 if r % 2 else 0)
            y = -r * dy
            return RIGHT * x + UP * y

        def make_hexagon():
            hexagon = RegularPolygon(
                n=6,
                radius=radius,
                color=WHITE,
                stroke_width=stroke_width,
            )
            hexagon.rotate(tilt, about_point=centroid(hexagon))
            return hexagon

        initial = [hex_center(r, c) for r in range(rows) for c in range(cols)]
        shift = -sum(initial, ORIGIN) / len(initial)
        initial = [pos + shift for pos in initial]
        initial_keys = {(r, c) for r in range(rows) for c in range(cols)}

        label = Text("Hexágonos", font_size=40, color="#E8D4A8")
        label.to_edge(UP, buff=0.4)
        self.play(FadeIn(label), run_time=0.6)

        for target in initial:
            hexagon = make_hexagon()
            hexagon.rotate(tilt, about_point=centroid(hexagon))

            start = RIGHT * start_x + UP * target[1]
            near = target + RIGHT * (1.2 * radius)
            move_centroid_to(hexagon, start)
            self.add(hexagon)

            self.play(
                hexagon.animate.shift(near - start),
                run_time=0.06,
                rate_func=rate_functions.ease_in_cubic,
            )

            hexagon.save_state()
            c0 = centroid(hexagon).copy()

            def land(mob, alpha, _c0=c0, _target=target):
                mob.restore()
                mob.shift(alpha * (_target - _c0))
                mob.rotate(
                    -tilt * alpha,
                    about_point=centroid(mob),
                )

            self.play(
                UpdateFromAlphaFunc(hexagon, land),
                run_time=0.16,
                rate_func=rate_functions.ease_out_cubic,
            )

        margin_x = config.frame_width / 2 + dx
        margin_y = config.frame_height / 2 + dy
        rest = []
        for r in range(-10, 14):
            for c in range(-10, 14):
                if (r, c) in initial_keys:
                    continue
                pos = hex_center(r, c) + shift
                if abs(pos[0]) > margin_x or abs(pos[1]) > margin_y:
                    continue
                hexagon = make_hexagon()
                move_centroid_to(hexagon, pos)
                rest.append(hexagon)

        rest.sort(key=lambda mob: np.linalg.norm(centroid(mob)))
        self.play(FadeOut(label), run_time=0.4)
        self.play(
            LaggedStart(*[Create(hexagon) for hexagon in rest], lag_ratio=0.02),
            run_time=2,
        )
        self.wait()


Manim Community v0.21.0

## Pentágonos

In [10]:
%%manim -qm PentagonTessellationFlyIn

class PentagonTessellationFlyIn(Scene):
    def construct(self):
        radius = 0.7
        stroke_width = 4
        tilt = 36 * DEGREES
        start_x = config.frame_width / 2 + 2 * radius
        gap_color = "#E8D4A8"

        alpha = PI / 5
        a = radius * np.cos(2 * alpha)
        b = radius * np.sin(2 * alpha)
        c = radius - a
        h = radius * np.cos(alpha)
        L = 2 * radius * np.sin(alpha)
        r1 = 2 * radius * np.sin(alpha / 2)
        r2 = 2 * radius * (
            np.cos(alpha) * (np.cos(alpha / 2) * np.tan(alpha) + np.sin(alpha / 2))
        )
        x = 2 * radius * np.cos(alpha) * np.sin(alpha / 2)
        y = 2 * radius * np.sin(alpha) * np.sin(alpha / 2)
        period = 4 * b + 3 * L + 2 * y

        def centroid(mob):
            return mob.get_vertices().mean(axis=0)

        def move_centroid_to(mob, point):
            mob.shift(point - centroid(mob))

        def xy_polygon(coords, **kwargs):
            points = [np.array([px, py, 0.0]) for px, py in coords]
            return Polygon(*points, **kwargs)

        def make_pentagon(pointing_up):
            pent = RegularPolygon(
                n=5,
                radius=radius,
                color=WHITE,
                stroke_width=stroke_width,
            )
            if not pointing_up:
                pent.rotate(PI, about_point=centroid(pent))
            return pent

        def make_star(pointing_down):
            coords = []
            for i in range(10):
                ang = PI * i / 5
                rad = r2 if i % 2 == 0 else r1
                coords.append((rad * np.cos(ang), rad * np.sin(ang)))
            star = xy_polygon(
                coords,
                color=gap_color,
                fill_color=gap_color,
                fill_opacity=0.9,
                stroke_width=2,
            )
            star.rotate(
                (-90 if pointing_down else 90) * DEGREES,
                about_point=centroid(star),
            )
            return star

        def make_rhombus():
            coords = [
                (-L / 2, -h),
                (L / 2, -h),
                (b + L / 2, -h - c),
                (b - L / 2, -h - c),
            ]
            return xy_polygon(
                coords,
                color=gap_color,
                fill_color=gap_color,
                fill_opacity=0.9,
                stroke_width=2,
            )

        pentagon_specs = [
            (True, 0.0, 0.0),
            (False, b + L / 2, a - h),
            (False, b, a + radius),
            (True, b + L + y, -radius - h),
            (True, b, radius + a + 2 * h),
            (False, b, -(c + 2 * h)),
            (False, b + L + y, radius + a + h + a),
            (False, b + L + y, -(3 * h + a + c)),
            (False, 3 * b + L / 2, a - h),
            (False, b + 2 * L + 2 * y, a + radius),
            (True, 3 * b + L, -(h + c + h + 2 * a)),
            (True, b + 2 * L + 2 * y, radius + a + 2 * h),
            (False, 2 * b + 2 * L + 2 * y, -2 * h),
            (True, 2 * b + 2 * L + 2 * y, 0),
            (True, 5 * b + L, -(h + c + h + 2 * a)),
            (False, 3 * b + 2 * L + 2 * y, a + radius),
            (True, 3 * b + 2 * L + 2 * y, radius + a + 2 * h),
            (False, 5 * b + 2 * L + y, -(3 * h + a + c)),
            (False, 5 * b + L / 2 + L, a - h),
            (True, 4 * b + 2 * L + L / 2 + 2 * y, radius + h),
            (True, 5 * b + 2 * L + y, -radius - h),
        ]

        def gap_mobjects():
            gaps = []
            rh = make_rhombus()
            gaps.append(rh.copy())

            r1_poly = xy_polygon(
                [
                    (b + L / 2 + 2 * b, -h - c),
                    (4 * b + L / 2, -h),
                    (b + 2 * L + 2 * y, -1 * a - 2 * h),
                    (L + 2 * b, -2 * h - c - a),
                ],
                color=gap_color,
                fill_color=gap_color,
                fill_opacity=0.9,
                stroke_width=2,
            )
            gaps.append(r1_poly)

            r2 = make_rhombus()
            r2.rotate(-72 * DEGREES, about_point=centroid(r2))
            r2.shift(
                RIGHT * (4 * b + L / 2 + y * np.cos(2 * alpha))
                + UP * (2 * h + 2 * a + c + y * np.sin(2 * alpha))
            )
            gaps.append(r2)

            r3_poly = xy_polygon(
                [
                    (4 * b + L / 2 + L, -h),
                    (5 * b + L / 2 + L, -h - c),
                    (6 * b + L, -2 * h - c - a),
                    (5 * b + L, -2 * h - a),
                ],
                color=gap_color,
                fill_color=gap_color,
                fill_opacity=0.9,
                stroke_width=2,
            )
            gaps.append(r3_poly)

            r4_poly = xy_polygon(
                [
                    (2 * b + 2 * L + 2 * y + b, a),
                    (3 * b + 2 * L + 2 * y + b, a + c),
                    (3 * b + 3 * L + 2 * y + b, a + c),
                    (2 * b + 3 * L + 2 * y + b, a),
                ],
                color=gap_color,
                fill_color=gap_color,
                fill_opacity=0.9,
                stroke_width=2,
            )
            gaps.append(r4_poly)

            s2 = make_star(True)
            move_centroid_to(s2, RIGHT * b + UP * (-(h + c + h + radius + r1)))
            gaps.append(s2)

            s3 = make_star(False)
            move_centroid_to(
                s3,
                RIGHT * (2 * b + 2 * L + 2 * y)
                + UP * (-(h + c + h + radius + r1 + a)),
            )
            gaps.append(s3)
            return gaps

        centers = [RIGHT * px + UP * py for _, px, py in pentagon_specs]
        shift = -sum(centers, ORIGIN) / len(centers)

        label = Text("¿Qué pasaría con pentágonos?", font_size=36, color="#E8D4A8")
        if label.width > config.frame_width - 1:
            label.scale_to_fit_width(config.frame_width - 1)
        label.to_edge(UP, buff=0.4)
        self.play(FadeIn(label), run_time=0.6)

        initial = []
        for pointing_up, px, py in pentagon_specs:
            target = RIGHT * px + UP * py + shift
            pent = make_pentagon(pointing_up)
            pent.rotate(tilt, about_point=centroid(pent))
            start = RIGHT * start_x + UP * target[1]
            near = target + RIGHT * (1.2 * radius)
            move_centroid_to(pent, start)
            self.add(pent)

            self.play(
                pent.animate.shift(near - start),
                run_time=0.06,
                rate_func=rate_functions.ease_in_cubic,
            )

            pent.save_state()
            c0 = centroid(pent).copy()

            def land(mob, alpha_t, _c0=c0, _target=target):
                mob.restore()
                mob.shift(alpha_t * (_target - _c0))
                mob.rotate(
                    -tilt * alpha_t,
                    about_point=centroid(mob),
                )

            self.play(
                UpdateFromAlphaFunc(pent, land),
                run_time=0.16,
                rate_func=rate_functions.ease_out_cubic,
            )
            initial.append(pent)

        margin_x = config.frame_width / 2 + 2 * radius
        margin_y = config.frame_height / 2 + 2 * radius
        rest = []
        gaps = []
        for n in range(-4, 5):
            origin = RIGHT * (n * period) + shift
            if n != 0:
                for pointing_up, px, py in pentagon_specs:
                    pos = origin + RIGHT * px + UP * py
                    if abs(pos[0]) > margin_x or abs(pos[1]) > margin_y:
                        continue
                    pent = make_pentagon(pointing_up)
                    move_centroid_to(pent, pos)
                    rest.append(pent)
            for gap in gap_mobjects():
                gap.shift(origin)
                pos = centroid(gap)
                if abs(pos[0]) > margin_x or abs(pos[1]) > margin_y:
                    continue
                gaps.append(gap)

        rest.sort(key=lambda mob: np.linalg.norm(centroid(mob)))
        gaps.sort(key=lambda mob: np.linalg.norm(centroid(mob)))

        closing = Text("siempre quedan huecos", font_size=36, color="#E8D4A8")
        if closing.width > config.frame_width - 1:
            closing.scale_to_fit_width(config.frame_width - 1)
        closing.to_edge(UP, buff=0.4)
        title_bottom = closing.get_bottom()[1] - 0.25
        title_left = closing.get_left()[0] - 0.4
        title_right = closing.get_right()[0] + 0.4
        gaps = [
            gap
            for gap in gaps
            if not (
                gap.get_top()[1] > title_bottom
                and gap.get_left()[0] < title_right
                and gap.get_right()[0] > title_left
            )
        ]

        self.play(FadeOut(label), run_time=0.4)
        if rest:
            self.play(
                LaggedStart(*[Create(pent) for pent in rest], lag_ratio=0.02),
                run_time=2,
            )
        self.play(
            LaggedStart(*[FadeIn(gap) for gap in gaps], lag_ratio=0.04),
            run_time=1.6,
        )
        self.play(FadeIn(closing), run_time=0.8)
        self.wait()


Manim Community v0.21.0